# Day 13: Overfitting & Regularization — Preventing Models from Memorizing Data

Welcome to Day 13 of the **60-Day Data Science Challenge**! 📈
Yesterday (Day 12), we built a baseline **Linear Regression** model to predict retail transaction sales. Today, we focus on **Model Optimization**, tackling one of the most critical challenges in machine learning: **Overfitting**.

## 🎯 Learning Objectives:
1. **Analyze Overfitting**: Identify when a model is learning the training data noise instead of the underlying pattern.
2. **Implement Ridge Regression (L2 Regularization)**: Apply L2 penalties to shrink model coefficients and reduce model variance.
3. **Implement Lasso Regression (L1 Regularization)**: Apply L1 penalties to perform automatic feature selection by driving non-essential coefficients to exactly zero.
4. **Analyze Train vs. Test Curves**: Evaluate how training error increases and testing error decreases as regularization is tuned.
5. **Interpret Regularization Paths**: Visualize coefficient shrinkage as a function of regularization strength ($\alpha$).

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set styling for professional visuals
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

### 1. Ingest & Inspect the Dataset
Let's load the engineered retail transactions dataset from Day 10 (`engineered_store_transactions.csv`) and verify its layout.

In [2]:
# Load dataset
df = pd.read_csv('../day10/engineered_store_transactions.csv')
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head(3)

### 2. Define features & Prevent Target Leakage
We must define our continuous target `Sales` and remove features that introduce target leakage or redundancy, matching the Day 12 pipeline.

In [3]:
# Continuous target variable
y = df['Sales']

# Exclude target leakage and identifiers
leakage_cols = [
    'Row ID', 'Order ID', 'Order Date', 'Customer Name', 
    'Sales', 'Sales_log', 'Sales_log_scaled', 
    'Sales_per_Unit', 'Sales_per_Unit_log', 'Sales_per_Unit_log_scaled',
    'Quantity', 'Order_Month', 'Order_Year', 'Order_DayOfWeek'
]

feature_cols = [col for col in df.columns if col not in leakage_cols]
X = df[feature_cols]

print(f"Target: 'Sales'")
print(f"\nWe are training our regression model on the following {len(feature_cols)} features:")
print(feature_cols)

### 3. Split Dataset into Training and Testing Sets
Using an 80/20 split locked with `random_state=42` to guarantee identical partitions as Day 12.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42
)
print(f"Training dimensions: {X_train.shape[0]} rows, {X_train.shape[1]} features.")
print(f"Testing dimensions:  {X_test.shape[0]} rows, {X_test.shape[1]} features.")

### 4. Train a Baseline OLS Linear Regression Model
Let's train a standard Ordinary Least Squares (OLS) model and calculate its performance on both training and test sets to establish a baseline.

In [5]:
ols = LinearRegression()
ols.fit(X_train, y_train)

y_train_pred_ols = ols.predict(X_train)
y_test_pred_ols = ols.predict(X_test)

train_r2_ols = r2_score(y_train, y_train_pred_ols)
test_r2_ols = r2_score(y_test, y_test_pred_ols)
train_rmse_ols = np.sqrt(mean_squared_error(y_train, y_train_pred_ols))
test_rmse_ols = np.sqrt(mean_squared_error(y_test, y_test_pred_ols))

print("--- BASELINE OLS REGRESSION METRICS ---")
print(f"Train R² Score: {train_r2_ols:.6f}")
print(f"Test R² Score: {test_r2_ols:.6f}")
print(f"Train RMSE:    ${train_rmse_ols:.4f}")
print(f"Test RMSE:     ${test_rmse_ols:.4f}")

#### 💡 Observation on Overfitting:
The baseline model exhibits a classic overfitting pattern: it has a positive R² score on the training set (**0.0157**), but a negative R² score on the test set (**-0.0257**).
Because the dataset is synthetic and lacks physical correlation, the model starts "memorizing" random fluctuations in the training partition (fitting noise), which fails to generalize to unseen test observations. We will now apply L2 (Ridge) and L1 (Lasso) regularization to combat this overfitting.

### 5. Train Ridge Regression Models (L2 Regularization)
Ridge regression adds an L2 penalty ($Penalty = \alpha \sum \beta_j^2$) to the loss function. This forces coefficients towards zero but never drives them to exactly zero. Let's analyze how varying $\alpha$ affects performance.

In [6]:
ridge_alphas = [0.1, 1.0, 10.0, 100.0, 500.0, 1000.0]
ridge_coefs = []
ridge_train_r2 = []
ridge_test_r2 = []
ridge_train_rmse = []
ridge_test_rmse = []

for a in ridge_alphas:
    model = Ridge(alpha=a, random_state=42)
    model.fit(X_train, y_train)
    ridge_coefs.append(model.coef_)
    
    tr_pred = model.predict(X_train)
    te_pred = model.predict(X_test)
    
    ridge_train_r2.append(r2_score(y_train, tr_pred))
    ridge_test_r2.append(r2_score(y_test, te_pred))
    ridge_train_rmse.append(np.sqrt(mean_squared_error(y_train, tr_pred)))
    ridge_test_rmse.append(np.sqrt(mean_squared_error(y_test, te_pred)))

print("Ridge models trained across a spectrum of alpha values!")

### 6. Train Lasso Regression Models (L1 Regularization)
Lasso regression adds an L1 penalty ($Penalty = \alpha \sum |\beta_j|$) to the loss function. This penalty is capable of shrinking coefficients to exactly zero, performing automatic feature selection.

In [7]:
lasso_alphas = [0.01, 0.1, 1.0, 5.0, 10.0, 20.0]
lasso_coefs = []
lasso_train_r2 = []
lasso_test_r2 = []
lasso_train_rmse = []
lasso_test_rmse = []

for a in lasso_alphas:
    model = Lasso(alpha=a, max_iter=10000, random_state=42)
    model.fit(X_train, y_train)
    lasso_coefs.append(model.coef_)
    
    tr_pred = model.predict(X_train)
    te_pred = model.predict(X_test)
    
    lasso_train_r2.append(r2_score(y_train, tr_pred))
    lasso_test_r2.append(r2_score(y_test, te_pred))
    lasso_train_rmse.append(np.sqrt(mean_squared_error(y_train, tr_pred)))
    lasso_test_rmse.append(np.sqrt(mean_squared_error(y_test, te_pred)))

print("Lasso models trained across a spectrum of alpha values!")

### 7. Visualize Coefficient Shrinkage Paths
Let's plot how coefficients shrink towards zero for both models as the regularization strength ($\alpha$) increases. This visually demonstrates L2 (proportional shrinkage) vs. L1 (exact zero sparsity).

In [8]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Ridge Coefficient Shrinkage Path
ridge_coefs_arr = np.array(ridge_coefs)
for i in range(len(feature_cols)):
    ax1.plot(ridge_alphas, ridge_coefs_arr[:, i], marker='o', label=feature_cols[i] if i < 5 else "")
ax1.set_xscale('log')
ax1.set_title('Ridge Coefficient Path (L2 Regularization)')
ax1.set_xlabel('Alpha (Log Scale)')
ax1.set_ylabel('Coefficient Dollar Impact')
ax1.grid(True, which="both", ls="--")

# Lasso Coefficient Shrinkage Path
lasso_coefs_arr = np.array(lasso_coefs)
for i in range(len(feature_cols)):
    ax2.plot(lasso_alphas, lasso_coefs_arr[:, i], marker='s', label=feature_cols[i] if i < 5 else "")
ax2.set_xscale('log')
ax2.set_title('Lasso Coefficient Path (L1 Regularization)')
ax2.set_xlabel('Alpha (Log Scale)')
ax2.set_ylabel('Coefficient Dollar Impact')
ax2.grid(True, which="both", ls="--")
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('coefficient_paths.png', dpi=150)
plt.show()
print("Saved coefficient_paths.png successfully!")

### 8. Visualize Train vs. Test Performance
Let's plot R² performance across different $\alpha$ values to understand how regularization combats overfitting. Notice how test R² gets pulled closer to 0 (mean predictor) from OLS's overfit negative value.

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Ridge R2 comparison
ax1.plot(ridge_alphas, ridge_train_r2, 'o--', label='Train R²', color='royalblue')
ax1.plot(ridge_alphas, ridge_test_r2, 's-', label='Test R²', color='darkorange')
ax1.set_xscale('log')
ax1.set_title('Ridge Performance vs. Alpha')
ax1.set_xlabel('Alpha (Log Scale)')
ax1.set_ylabel('R² Score')
ax1.legend()
ax1.grid(True, which="both", ls="--")

# Lasso R2 comparison
ax2.plot(lasso_alphas, lasso_train_r2, 'o--', label='Train R²', color='royalblue')
ax2.plot(lasso_alphas, lasso_test_r2, 's-', label='Test R²', color='darkorange')
ax2.set_xscale('log')
ax2.set_title('Lasso Performance vs. Alpha')
ax2.set_xlabel('Alpha (Log Scale)')
ax2.set_ylabel('R² Score')
ax2.legend()
ax2.grid(True, which="both", ls="--")

plt.tight_layout()
plt.savefig('train_test_performance.png', dpi=150)
plt.show()
print("Saved train_test_performance.png successfully!")

### 9. Automatic Feature Selection with Lasso (Alpha=5)
Lasso's ability to drive coefficients to exactly zero allows us to remove feature noise completely. Let's see which features Lasso preserved at $\alpha=5$.

In [10]:
opt_lasso = Lasso(alpha=5.0, max_iter=10000, random_state=42)
opt_lasso.fit(X_train, y_train)

print("--- LASSO FEATURE SELECTION (ALPHA=5) ---")
print("Features preserved:")
for col, coef in zip(feature_cols, opt_lasso.coef_):
    if coef != 0.0:
        print(f"  {col}: {coef:.4f}")

print("\nFeatures eliminated (driven to exactly zero):")
for col, coef in zip(feature_cols, opt_lasso.coef_):
    if coef == 0.0:
        print(f"  {col}: {coef:.4f}")

### 10. Compare Model Performance Metrics
Let's compare the performance of OLS, Ridge ($\alpha=100$), and Lasso ($\alpha=5$) on both Train and Test datasets.

In [11]:
opt_ridge = Ridge(alpha=100.0, random_state=42)
opt_ridge.fit(X_train, y_train)
y_train_pred_ridge = opt_ridge.predict(X_train)
y_test_pred_ridge = opt_ridge.predict(X_test)

metrics = {
    'OLS': {
        'Train MAE': mean_absolute_error(y_train, y_train_pred_ols),
        'Test MAE': mean_absolute_error(y_test, y_test_pred_ols),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred_ols)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred_ols)),
        'Train R²': r2_score(y_train, y_train_pred_ols),
        'Test R²': r2_score(y_test, y_test_pred_ols)
    },
    'Ridge (alpha=100)': {
        'Train MAE': mean_absolute_error(y_train, y_train_pred_ridge),
        'Test MAE': mean_absolute_error(y_test, y_test_pred_ridge),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred_ridge)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred_ridge)),
        'Train R²': r2_score(y_train, y_train_pred_ridge),
        'Test R²': r2_score(y_test, y_test_pred_ridge)
    },
    'Lasso (alpha=5)': {
        'Train MAE': mean_absolute_error(y_train, y_train_pred_lasso),
        'Test MAE': mean_absolute_error(y_test, y_test_pred_lasso),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred_lasso)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred_lasso)),
        'Train R²': r2_score(y_train, y_train_pred_lasso),
        'Test R²': r2_score(y_test, y_test_pred_lasso)
    }
}

metrics_df = pd.DataFrame(metrics).T
print(metrics_df.to_string(formatters={
    'Train MAE': '{:,.4f}'.format,
    'Test MAE': '{:,.4f}'.format,
    'Train RMSE': '{:,.4f}'.format,
    'Test RMSE': '{:,.4f}'.format,
    'Train R²': '{:,.6f}'.format,
    'Test R²': '{:,.6f}'.format,
}))

#### 💡 Conclusion on Regularization Dynamics:
- **OLS** has the highest Train R² (**0.0157**) but the worst Test R² (**-0.0257**), indicating severe overfitting to training noise.
- **Ridge** shrinks feature slopes, reducing the Train R² to **0.0137**, but significantly improving the Test R² to **-0.0162** and dropping Test RMSE from $404.06 to $402.19.
- **Lasso** performs even better by stripping out 9 irrelevant features, achieving a Test R² of **-0.0103** and the lowest Test RMSE of **$401.02**.

By adding regularization, we reduce variance, shrink the generalization gap, and pull test performance back toward the baseline mean ($R^2 = 0.0$).

### 11. Export predictions to CSV
Finally, let's export actual test sales alongside the predictions and residuals of all three models to `predictions_regularization.csv`.

In [12]:
pred_df = pd.DataFrame({
    'Actual_Sales': y_test,
    'OLS_Predicted': y_test_pred_ols,
    'Ridge_Predicted': y_test_pred_ridge,
    'Lasso_Predicted': y_test_pred_lasso,
    'OLS_Residual': y_test - y_test_pred_ols,
    'Ridge_Residual': y_test - y_test_pred_ridge,
    'Lasso_Residual': y_test - y_test_pred_lasso
}).reset_index(drop=True)

pred_df.to_csv('predictions_regularization.csv', index=False)
print("Successfully exported predictions_regularization.csv!")